# Hebrew lecture → Whisper large-v3 transcript → Russian translation (GPT)

Self-contained **Google Colab** notebook (independent of the main pipeline):

1. Download the Hebrew audio from Google Drive.
2. Transcribe it with **faster-whisper `large-v3`** (Hebrew), keeping per-segment timecodes.
3. Translate every segment to **Russian** with **`gpt-5.4-nano`** (OpenAI API). For each line we
   send GPT a **sliding window of the last 3 subtitle lines** for context and translate only the
   **last** one — the window then slides forward by one line.
4. The Russian output keeps **exactly the same timecodes** as the Hebrew (1:1 per segment).

**Before running:**
- Runtime → Change runtime type → Hardware accelerator = **GPU** (T4 is enough, for Whisper).
- Add your OpenAI key as a Colab **Secret** named `OPENAI_API_KEY` (🔑 panel, left sidebar) and
  toggle *Notebook access* on.
- The Drive file must be shared "Anyone with the link".


In [ ]:
# 1. Install dependencies (~1-2 min on a fresh Colab VM).
!pip install -q faster-whisper gdown openai
print("deps ready")


In [ ]:
# 2. Download the Hebrew audio from Google Drive.
import gdown, os

GDRIVE_URL = "https://drive.google.com/file/d/1hw8WzVpmi1pMyEfQcLp2jl166YdDvCKs/view?usp=sharing"

# fuzzy=True lets gdown pull the file id out of a full share URL and keep the original name.
MEDIA_PATH = gdown.download(url=GDRIVE_URL, output="lecture_input", fuzzy=True, quiet=False)
assert MEDIA_PATH and os.path.exists(MEDIA_PATH), "download failed - is the file shared with 'Anyone with the link'?"
print("downloaded:", MEDIA_PATH, f"({os.path.getsize(MEDIA_PATH)/1e6:.1f} MB)")


In [ ]:
# 3. Transcribe with faster-whisper large-v3 (Hebrew), keep per-segment timecodes.
#    faster-whisper decodes the media directly (audio OR video) via PyAV - no separate ffmpeg step.
import gc, torch
from faster_whisper import WhisperModel

device  = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"loading whisper large-v3 on {device}/{compute} ...")
asr = WhisperModel("large-v3", device=device, compute_type=compute)

segments_iter, info = asr.transcribe(
    MEDIA_PATH,
    language="he",        # force Hebrew (the lecture language)
    vad_filter=True,      # drop silence so timecodes hug actual speech
    word_timestamps=False,
)
print(f"detected language={info.language} (p={info.language_probability:.2f}), starting decode ...")

he_segments = []  # each: {"id", "start", "end", "text"}
for i, s in enumerate(segments_iter):
    text = s.text.strip()
    he_segments.append({"id": i, "start": round(s.start, 3), "end": round(s.end, 3), "text": text})
    print(f"  {s.start:7.1f}s  {text[:80]}")

print(f"\n{len(he_segments)} segments transcribed.")

# free the Whisper model once we have the segments (translation runs via API, no GPU needed).
del asr
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 3b. Save the Hebrew transcript (JSON + SRT + readable TXT with timecodes).
import json
from pathlib import Path

OUT = Path("output"); OUT.mkdir(exist_ok=True)

def srt_time(t):
    h, rem = divmod(int(t), 3600); m, s = divmod(rem, 60); ms = int((t - int(t)) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def clock(t):
    h, rem = divmod(int(t), 3600); m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def write_outputs(segments, stem):
    (OUT / f"{stem}.json").write_text(json.dumps(segments, ensure_ascii=False, indent=2), "utf-8")
    srt = []
    for n, seg in enumerate(segments, 1):
        srt += [str(n), f"{srt_time(seg['start'])} --> {srt_time(seg['end'])}", seg["text"], ""]
    (OUT / f"{stem}.srt").write_text("\n".join(srt), "utf-8")
    txt = [f"[{clock(seg['start'])}-{clock(seg['end'])}] {seg['text']}" for seg in segments]
    (OUT / f"{stem}.txt").write_text("\n".join(txt), "utf-8")

write_outputs(he_segments, "transcript_he")
print("wrote output/transcript_he.{json,srt,txt}")


In [ ]:
# 4. Set up the GPT translator (OpenAI gpt-5.4-nano).
#    Key comes from a Colab Secret named OPENAI_API_KEY.
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-5.4-nano"

# Domain context: telling the model it's a databases lecture keeps SQL/relational terms accurate.
SYSTEM = (
    "You are a professional subtitle translator. You translate a university lecture about "
    "DATABASES from Hebrew into Russian. The lecture covers relational databases, SQL, the "
    "relational model, keys (primary/foreign), normalization and normal forms, ER diagrams, "
    "transactions and ACID, indexing, joins, and query optimization. "
    "Translate faithfully and fluently into natural Russian, using the established Russian "
    "database terms (e.g. 'внешний ключ', 'нормальная форма', 'транзакция'). Keep SQL keywords, "
    "table/column names, and mathematical notation unchanged. "
    "You are given a few consecutive subtitle lines for CONTEXT, but you must translate ONLY the "
    "last line. Output just the Russian translation of that last line - no notes, no numbering, "
    "no original Hebrew."
)

def translate_window(lines):
    """lines = up to 3 consecutive Hebrew lines; translate only the LAST one (the rest are context)."""
    block = "\n".join(f"{i+1}. {t}" for i, t in enumerate(lines))
    user = (
        "Consecutive Hebrew subtitle lines from the lecture (in order). "
        f"Translate ONLY line {len(lines)} (the last one) into Russian; use the earlier lines for "
        "context. Reply with the Russian translation of the last line only.\n\n" + block
    )
    last_err = None
    for attempt in range(4):  # simple retry for transient API/rate errors
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": user}],
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            last_err = e
            import time; time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"translation failed after retries: {last_err}")

print("translator ready:", MODEL)


In [ ]:
# 5. Translate He -> Ru with a sliding 3-line context window (strict 1:1, timecodes preserved).
#    For segment i we send GPT lines [i-2, i-1, i] and translate only line i. The window then
#    slides forward by one, so every line is translated exactly once with its local context.
ru_segments = []
for i, seg in enumerate(he_segments):
    window = he_segments[max(0, i - 2): i + 1]   # last up to 3 lines; target = the last one
    ru = translate_window([s["text"] for s in window])
    ru_segments.append({"id": seg["id"], "start": seg["start"], "end": seg["end"], "text": ru})
    print(f"  [{clock(seg['start'])}] {ru[:80]}")
    if (i + 1) % 25 == 0:
        print(f"... {i + 1}/{len(he_segments)} translated")

assert len(ru_segments) == len(he_segments), "segment count drifted - timecodes would misalign"
print("\ndone:", len(ru_segments), "Russian segments (timecodes preserved 1:1)")


In [ ]:
# 5b. Save the Russian translation + a side-by-side bilingual file (same timecodes).
write_outputs(ru_segments, "transcript_ru")

bilingual = []
for he, ru in zip(he_segments, ru_segments):
    bilingual += [f"[{clock(he['start'])}-{clock(he['end'])}]", f"  HE: {he['text']}", f"  RU: {ru['text']}", ""]
(OUT / "transcript_bilingual.txt").write_text("\n".join(bilingual), "utf-8")

print("wrote:")
for p in sorted(OUT.glob("transcript_*")):
    print("  ", p)


In [ ]:
# 6. (Optional) Download all outputs as a single zip.
import shutil
shutil.make_archive("transcripts", "zip", "output")
from google.colab import files
files.download("transcripts.zip")


**Outputs** (in `output/`, also zipped by the last cell):

| file | what |
|------|------|
| `transcript_he.{json,srt,txt}` | Hebrew transcript with timecodes |
| `transcript_ru.{json,srt,txt}` | Russian translation, **same timecodes** |
| `transcript_bilingual.txt` | He/Ru side by side per timecode |

The `.srt` files drop straight into a video player as subtitles. The `.json` files keep
`start`/`end` seconds per segment if you need to process them further.
